# Biblioteca comum — Pós-processamento e métricas

Funções reutilizáveis extraídas de `01_academic/04_reference_materials/03_code_exemple/Pos_Metrics.ipynb`:

- `pos_process()` — opening morfológico + componente conexo principal (labeling)
- `calculate_metrics()` — Dice, Accuracy, Precision, Recall
- Emparelhamento **imagem/previsão ↔ label** por identificador base (`P01`, `P15`, …)
- Validação de paths e pastas do projeto

Consumido por `03_pipeline/04_evaluation/evaluation.ipynb` (e opcionalmente por segmentação via `%run`).

## Imports

In [ ]:
%matplotlib inline

import re
from pathlib import Path

import numpy as np
import matplotlib.image as mpimg
from scipy.ndimage import binary_opening, label

## Paths do projeto (sem Colab / Google Drive)

In [ ]:
def encontrar_raiz_projeto(inicio: Path) -> Path:
    """Sobe na árvore até encontrar 02_dataset/ (funciona em qualquer subpasta do repo)."""
    for candidato in [inicio, *inicio.parents]:
        if (candidato / "02_dataset").is_dir():
            return candidato
    raise FileNotFoundError(
        f"Pasta 02_dataset/ não encontrada a partir de {inicio}. "
        "Execute o notebook a partir do repositório fetal_vein_segmentation."
    )


PROJECT_ROOT = encontrar_raiz_projeto(Path.cwd().resolve())
DATA_DIR = PROJECT_ROOT / "02_dataset"
IMAGES_ORIGINAL_DIR = DATA_DIR / "images"
LABELS_DIR = DATA_DIR / "labels"
SAVE_MODELS_DIR = DATA_DIR / "Save_Models"

PASTAS_IMAGENS_PP = [
    DATA_DIR / "images_pp_1",
    DATA_DIR / "images_pp_2",
    DATA_DIR / "images_pp_3",
    DATA_DIR / "images_pp_4",
    DATA_DIR / "images_pp_5",
]

PASTAS_RESULTADOS = [
    ("Original", DATA_DIR / "results_original"),
    ("PP1", DATA_DIR / "results_pp_1"),
    ("PP2", DATA_DIR / "results_pp_2"),
    ("PP3", DATA_DIR / "results_pp_3"),
    ("PP4", DATA_DIR / "results_pp_4"),
    ("PP5", DATA_DIR / "results_pp_5"),
]

MODELOS_ESPERADOS = [
    "best_metric_model_original.pth",
    "best_metric_model_pp_1.pth",
    "best_metric_model_pp_2.pth",
    "best_metric_model_pp_3.pth",
    "best_metric_model_pp_4.pth",
    "best_metric_model_pp_5.pth",
]

EXTENSOES_IMAGEM = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

## Identificador base e emparelhamento (nunca por ordem na lista)

In [ ]:
BASE_ID_PATTERN = re.compile(r"^(P\d+)", re.IGNORECASE)


def extrair_identificador_base(nome_ficheiro: str) -> str:
    """Ex.: P01_PP_PL_1.png → P01; P15_PP_PL_4.png → P15."""
    stem = Path(nome_ficheiro).stem
    correspondencia = BASE_ID_PATTERN.match(stem)
    if correspondencia is None:
        raise ValueError(
            f"Identificador base não encontrado em '{nome_ficheiro}'. "
            f"Esperado prefixo P01, P15, P37, …"
        )
    return correspondencia.group(1).upper()


def resolver_caminho_label(nome_previsao: str, labels_dir: Path = LABELS_DIR) -> Path:
    """Associa previsão ao GT em labels/ pelo ID principal."""
    id_base = extrair_identificador_base(nome_previsao)
    candidatos = [
        labels_dir / f"{id_base}.png",
        labels_dir / f"{Path(nome_previsao).stem}.png",
    ]
    for candidato in candidatos:
        if candidato.is_file():
            return candidato
    raise FileNotFoundError(
        f"Label em falta para '{nome_previsao}' (ID {id_base}). "
        f"Candidatos: {[str(c) for c in candidatos]}"
    )


def listar_previsoes(pasta_resultados: Path) -> list:
    """Lista ficheiros de previsão ordenados por nome (não define emparelhamento)."""
    if not pasta_resultados.is_dir():
        return []
    return sorted(
        p
        for p in pasta_resultados.iterdir()
        if p.is_file() and p.suffix.lower() in EXTENSOES_IMAGEM
    )

## Validação de paths, datasets e emparelhamentos

In [ ]:
def validar_paths_projeto() -> list:
    """Verifica pastas obrigatórias; devolve lista de avisos (strings)."""
    avisos = []
    obrigatorias = [
        ("images (original)", IMAGES_ORIGINAL_DIR),
        ("labels", LABELS_DIR),
        ("Save_Models", SAVE_MODELS_DIR),
    ]
    for nome, caminho in obrigatorias:
        if not caminho.is_dir():
            avisos.append(f"Pasta em falta: {nome} → {caminho}")

    for indice, pasta in enumerate(PASTAS_IMAGENS_PP, start=1):
        if not pasta.is_dir():
            avisos.append(f"Dataset pré-processado em falta: images_pp_{indice}")
        elif len(list(pasta.glob("*.png"))) == 0:
            avisos.append(f"images_pp_{indice} existe mas não contém .png")

    for nome_exp, pasta in PASTAS_RESULTADOS:
        if not pasta.is_dir():
            avisos.append(f"Pasta de resultados em falta: {nome_exp} → {pasta.name}")
        elif len(listar_previsoes(pasta)) == 0:
            avisos.append(f"{pasta.name} existe mas não contém previsões")

    for nome_modelo in MODELOS_ESPERADOS:
        caminho = SAVE_MODELS_DIR / nome_modelo
        if not caminho.is_file():
            avisos.append(f"Modelo em falta: {nome_modelo}")

    return avisos


def validar_emparelhamento_pasta(pasta_imagens: Path, labels_dir: Path = LABELS_DIR) -> list:
    """Valida cada imagem/previsão → label por ID (não por índice)."""
    erros = []
    if not pasta_imagens.is_dir():
        return [f"Pasta inexistente: {pasta_imagens}"]
    for caminho in sorted(pasta_imagens.glob("*.png")):
        try:
            resolver_caminho_label(caminho.name, labels_dir)
        except (ValueError, FileNotFoundError) as exc:
            erros.append(f"{caminho.name}: {exc}")
    return erros


def validar_todos_emparelhamentos() -> dict:
    """Valida images, images_pp_* e pastas de resultados."""
    relatorio = {}
    relatorio["images"] = validar_emparelhamento_pasta(IMAGES_ORIGINAL_DIR)
    for indice, pasta in enumerate(PASTAS_IMAGENS_PP, start=1):
        relatorio[f"images_pp_{indice}"] = validar_emparelhamento_pasta(pasta)
    for nome_exp, pasta_res in PASTAS_RESULTADOS:
        erros = []
        for prev in listar_previsoes(pasta_res):
            try:
                resolver_caminho_label(prev.name)
            except (ValueError, FileNotFoundError) as exc:
                erros.append(f"{prev.name}: {exc}")
        relatorio[f"results_{nome_exp}"] = erros
    return relatorio

## Funções auxiliares de carregamento e orientação

A docente aplica correção de orientação às previsões antes das métricas (`Pos_Metrics.ipynb`).

In [ ]:
def carregar_mascara_png(caminho: Path) -> np.ndarray:
    """Carrega máscara PNG como array 2D."""
    imagem = mpimg.imread(caminho)
    if imagem.ndim == 3:
        imagem = imagem[:, :, 0]
    return imagem


def binarizar_mascara(mascara: np.ndarray, limiar: float = 127.0) -> np.ndarray:
    """Converte máscara para binária {0, 1} uint8."""
    if mascara.dtype == np.bool_:
        return mascara.astype(np.uint8)
    if mascara.max() <= 1.0:
        return (mascara > 0.5).astype(np.uint8)
    return (mascara >= limiar).astype(np.uint8)


def corrigir_orientacao_previsao(previsao: np.ndarray) -> np.ndarray:
    """
    Corrige orientação da previsão (equivalente ao notebook Pos_Metrics).
    predicted = np.flip(np.rot90(predicted, 1), 0)
    """
    return np.flip(np.rot90(previsao, 1), 0)

## `pos_process()` — pós-processamento morfológico

Implementação do esqueleto da docente: **opening** + **labeling** (mantém o maior componente conexo).

In [ ]:
def pos_process(predicted: np.ndarray, tamanho_opening: int = 3) -> np.ndarray:
    """
    Pós-processamento: opening morfológico + maior componente ligado.

    Args:
        predicted: máscara binária ou em escala de cinza.
        tamanho_opening: lado do elemento estruturante (quadrado).

    Returns:
        Máscara binária uint8 {0, 1}.
    """
    mascara = binarizar_mascara(predicted)
    estrutura = np.ones((tamanho_opening, tamanho_opening), dtype=bool)
    aberta = binary_opening(mascara > 0, structure=estrutura)

    rotulos, numero = label(aberta)
    if numero == 0:
        return np.zeros_like(mascara, dtype=np.uint8)

    maior_rotulo = 1
    maior_area = 0
    for rotulo in range(1, numero + 1):
        area = np.sum(rotulos == rotulo)
        if area > maior_area:
            maior_area = area
            maior_rotulo = rotulo

    predicted_pos_proc = (rotulos == maior_rotulo).astype(np.uint8)
    return predicted_pos_proc

## `calculate_metrics()` — Dice, Accuracy, Precision, Recall

In [ ]:
def calculate_metrics(predicted: np.ndarray, gt: np.ndarray):
    """
    Calcula métricas de segmentação binária.

    Returns:
        (dice, accuracy, precision, recall)
    """
    pred = binarizar_mascara(predicted)
    gt_bin = binarizar_mascara(gt)

    if pred.shape != gt_bin.shape:
        raise ValueError(
            f"Dimensões incompatíveis: previsão {pred.shape} vs GT {gt_bin.shape}"
        )

    tp = int(np.sum((pred == 1) & (gt_bin == 1)))
    tn = int(np.sum((pred == 0) & (gt_bin == 0)))
    fp = int(np.sum((pred == 1) & (gt_bin == 0)))
    fn = int(np.sum((pred == 0) & (gt_bin == 1)))

    eps = 1e-8
    dice = (2.0 * tp) / (2.0 * tp + fp + fn + eps)
    ac = (tp + tn) / (tp + tn + fp + fn + eps)
    pr = tp / (tp + fp + eps)
    re = tp / (tp + fn + eps)

    return dice, ac, pr, re


def calcular_metricas_medias(lista_metricas: list) -> dict:
    """Média de tuplos (dice, ac, pr, re)."""
    if len(lista_metricas) == 0:
        return {"dice": np.nan, "accuracy": np.nan, "precision": np.nan, "recall": np.nan}
    arr = np.array(lista_metricas)
    return {
        "dice": float(np.mean(arr[:, 0])),
        "accuracy": float(np.mean(arr[:, 1])),
        "precision": float(np.mean(arr[:, 2])),
        "recall": float(np.mean(arr[:, 3])),
    }

## Validação rápida (executar após importar biblioteca)

In [ ]:
print(f"Raiz do projeto: {PROJECT_ROOT}")
print("--- Avisos de paths/datasets ---")
for aviso in validar_paths_projeto():
    print(f"  [AVISO] {aviso}")
if not validar_paths_projeto():
    print("  Estrutura de pastas OK (ou sem avisos críticos listados).")

print("--- Emparelhamento images_pp_1 (exemplo) ---")
erros_pp1 = validar_emparelhamento_pasta(PASTAS_IMAGENS_PP[0])
if erros_pp1:
    for e in erros_pp1[:5]:
        print(f"  [ERRO] {e}")
    if len(erros_pp1) > 5:
        print(f"  … e mais {len(erros_pp1) - 5} erros")
else:
    print("  images_pp_1: emparelhamento por ID OK (ou pasta vazia/inexistente)")